# UBX Track Plotter
Parses `.ubx` log files and plots the latitude/longitude track.
Uses the same NAV-PVT + NAV-HPPOSLLH merge logic as `v3_ubx_parser.py`.

**Requirements**
```
pip install pyubx2 pandas matplotlib folium
```

In [ ]:
import glob
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors

try:
    import folium
    HAS_FOLIUM = True
except ImportError:
    HAS_FOLIUM = False
    print("folium not installed — interactive map will be skipped. Run: pip install folium")

from pyubx2 import UBXReader

## 1 — Select file
Set `UBX_FILE` to the path of your `.ubx` log, or leave it as `None` to auto-detect the most recent one in this directory.

In [ ]:
UBX_FILE = None   # e.g. "../data/dataLog00029.ubx"

if UBX_FILE is None:
    candidates = sorted(glob.glob("../**/*.ubx", recursive=True) + glob.glob("*.ubx"))
    if not candidates:
        sys.exit("No .ubx files found. Set UBX_FILE manually.")
    UBX_FILE = candidates[-1]   # most recent by name

print(f"Parsing: {UBX_FILE}")

## 2 — Parse

In [ ]:
def parse_ubx(path):
    """Returns a DataFrame of position records, one row per epoch."""
    pvt, hppos = {}, {}

    with open(path, "rb") as f:
        for _, parsed in UBXReader(f):
            if not hasattr(parsed, "identity"):
                continue
            iTOW = getattr(parsed, "iTOW", None)
            if iTOW is None:
                continue

            if parsed.identity == "NAV-PVT":
                pvt[iTOW] = dict(
                    iTOW=iTOW,
                    year=getattr(parsed, "year", 0),
                    month=getattr(parsed, "month", 0),
                    day=getattr(parsed, "day", 0),
                    hour=getattr(parsed, "hour", 0),
                    minute=getattr(parsed, "min", 0),
                    second=getattr(parsed, "sec", 0),
                    fix_type=getattr(parsed, "fixType", 0),
                    carrier_solution=getattr(parsed, "carrSoln", 0),
                    num_sv=getattr(parsed, "numSV", 0),
                    pdop=getattr(parsed, "pDOP", 0) / 100.0,
                    speed_ms=getattr(parsed, "gSpeed", 0) / 1000.0,
                    lat_pvt=getattr(parsed, "lat", 0) * 1e-7,
                    lon_pvt=getattr(parsed, "lon", 0) * 1e-7,
                    h_acc_pvt=getattr(parsed, "hAcc", 0) / 1000.0,
                )

            elif parsed.identity == "NAV-HPPOSLLH":
                lat  = getattr(parsed, "lat", 0) * 1e-7
                lon  = getattr(parsed, "lon", 0) * 1e-7
                hpLat = getattr(parsed, "latHp",  getattr(parsed, "hpLat",  0))
                hpLon = getattr(parsed, "lonHp",  getattr(parsed, "hpLon",  0))
                hppos[iTOW] = dict(
                    lat_hp=lat + hpLat * 1e-9,
                    lon_hp=lon + hpLon * 1e-9,
                    h_acc_hp=getattr(parsed, "hAcc", 0) / 10000.0,
                )

    use_hp = len(hppos) > 0
    print(f"NAV-PVT: {len(pvt)}   NAV-HPPOSLLH: {len(hppos)}   "
          f"→ using {'HPPOSLLH' if use_hp else 'PVT'} positions")

    rows = []
    for iTOW in sorted(pvt):
        p = pvt[iTOW].copy()
        hp = hppos.get(iTOW, {})
        if hp:
            p["latitude"]  = hp["lat_hp"]
            p["longitude"] = hp["lon_hp"]
            p["h_acc"]     = hp["h_acc_hp"]
            p["source"]    = "HP"
        else:
            p["latitude"]  = p["lat_pvt"]
            p["longitude"] = p["lon_pvt"]
            p["h_acc"]     = p["h_acc_pvt"]
            p["source"]    = "PVT"
        rows.append(p)

    df = pd.DataFrame(rows)
    df["datetime"] = pd.to_datetime(
        df[["year","month","day","hour","minute","second"]]
        .rename(columns={"minute":"minute","second":"second"}),
        errors="coerce"
    )
    return df


df = parse_ubx(UBX_FILE)
print(f"\n{len(df)} epochs loaded")
df[["datetime","latitude","longitude","h_acc","fix_type","carrier_solution","num_sv"]].head()

## 3 — Fix-quality summary

In [ ]:
FIX_NAMES = {0:"No Fix", 1:"DR", 2:"2D", 3:"3D", 4:"GNSS+DR", 5:"Time"}
RTK_NAMES = {0:"None", 1:"Float", 2:"Fixed"}

print("Fix types:")
print(df["fix_type"].map(FIX_NAMES).value_counts().to_string())
print("\nRTK carrier solution:")
print(df["carrier_solution"].map(RTK_NAMES).value_counts().to_string())
print(f"\nMean h_acc (all):       {df['h_acc'].mean():.3f} m")
rtk = df[df["carrier_solution"] == 2]
if len(rtk):
    print(f"Mean h_acc (RTK Fixed): {rtk['h_acc'].mean()*100:.1f} cm")

## 4 — Matplotlib track plot
Points are colored by RTK carrier solution status.

In [ ]:
RTK_COLORS = {0: "#d62728", 1: "#ff7f0e", 2: "#2ca02c"}   # red / orange / green
RTK_LABELS = {0: "No RTK", 1: "RTK Float", 2: "RTK Fixed"}

fig, ax = plt.subplots(figsize=(8, 6))

# draw track line in light grey first
ax.plot(df["longitude"], df["latitude"], color="#cccccc", linewidth=0.8, zorder=1)

for sol, grp in df.groupby("carrier_solution"):
    ax.scatter(
        grp["longitude"], grp["latitude"],
        c=RTK_COLORS.get(sol, "grey"),
        label=RTK_LABELS.get(sol, str(sol)),
        s=10, zorder=2, alpha=0.8
    )

# mark start / end
ax.plot(df["longitude"].iloc[0],  df["latitude"].iloc[0],  "k^", ms=8, label="Start")
ax.plot(df["longitude"].iloc[-1], df["latitude"].iloc[-1], "ks", ms=8, label="End")

ax.set_xlabel("Longitude (°)")
ax.set_ylabel("Latitude (°)")
ax.set_title(f"GPS Track — {Path(UBX_FILE).name}")
ax.legend(markerscale=1.5, fontsize=9)
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

## 5 — Horizontal accuracy over time

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))

ax.plot(df["iTOW"] / 1000.0, df["h_acc"] * 100, linewidth=0.8, color="steelblue")
ax.set_xlabel("GPS time of week (s)")
ax.set_ylabel("Horizontal accuracy (cm)")
ax.set_title("Horizontal accuracy over time")
ax.set_yscale("log")
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()

## 6 — Interactive map (folium)
Points colored by RTK status. Click any point to see timestamp and accuracy.

In [ ]:
if not HAS_FOLIUM:
    print("Install folium to enable this cell: pip install folium")
else:
    center = [df["latitude"].mean(), df["longitude"].mean()]
    fmap = folium.Map(location=center, zoom_start=16, tiles="OpenStreetMap")

    # track polyline
    coords = list(zip(df["latitude"], df["longitude"]))
    folium.PolyLine(coords, color="grey", weight=1.5, opacity=0.6).add_to(fmap)

    for _, row in df.iterrows():
        color = RTK_COLORS.get(int(row["carrier_solution"]), "grey")
        popup = (
            f"{row['datetime']}<br>"
            f"RTK: {RTK_LABELS.get(int(row['carrier_solution']), '?')}<br>"
            f"h_acc: {row['h_acc']*100:.1f} cm<br>"
            f"SVs: {int(row['num_sv'])}"
        )
        folium.CircleMarker(
            location=[row["latitude"], row["longitude"]],
            radius=3, color=color, fill=True, fill_opacity=0.8,
            popup=folium.Popup(popup, max_width=200)
        ).add_to(fmap)

    fmap